# NumPy Zero to Hero

A detailed, ground-up NumPy refresher: array creation through broadcasting, views vs. copies,
linear algebra, and performance — with the project's own `(N, 20, 13)` window tensor as a running
example so the abstract rules land somewhere concrete. Companion notebooks:
`refresher_pandas.ipynb`, `refresher_pytorch.ipynb`, and `refresher_deep_learning.ipynb` (concepts).

**Contents**
1. Why NumPy (vs. plain Python lists)
2. Creating arrays
3. dtype and casting
4. Shape, dimensions, reshaping
5. Indexing and slicing
6. Fancy indexing and boolean masking
7. Views vs. copies
8. Broadcasting, in depth
9. Vectorized math and ufuncs
10. Aggregations and the `axis` argument
11. Stacking, concatenating, splitting
12. Linear algebra
13. The `random` module
14. Sorting and searching
15. Performance: vectorization vs. loops
16. Saving and loading
17. Worked example: this project's window tensor
18. Common pitfalls, summarized


## 1. Why NumPy

A Python `list` stores pointers to arbitrary objects scattered in memory; a NumPy `ndarray` stores
one fixed-size dtype in one contiguous memory block. That single decision is why NumPy is fast:
operations run as tight, cache-friendly C loops (or SIMD instructions) instead of a Python
bytecode loop touching Python objects one at a time.


In [ ]:
import numpy as np
import time

n = 2_000_000
py_list = list(range(n))
np_array = np.arange(n)

t0 = time.perf_counter()
py_result = [x * 2 for x in py_list]
t_list = time.perf_counter() - t0

t0 = time.perf_counter()
np_result = np_array * 2
t_np = time.perf_counter() - t0

print(f"Python list comprehension: {t_list*1000:.1f} ms")
print(f"NumPy vectorized op:       {t_np*1000:.1f} ms")
print(f"speedup: {t_list / t_np:.0f}x")


## 2. Creating arrays

The most common constructors, roughly in order of how often you'll reach for them.


In [ ]:
import numpy as np

a = np.array([1, 2, 3])                       # from a Python list
b = np.array([[1, 2, 3], [4, 5, 6]])           # from nested lists -> 2D
zeros = np.zeros((2, 3))                       # shape-filled with 0.0
ones = np.ones((2, 3))
full = np.full((2, 3), 7)                      # shape filled with a constant
eye = np.eye(3)                                # identity matrix
arange = np.arange(0, 10, 2)                   # like range(), but an array: start, stop, step
linspace = np.linspace(0, 1, 5)                # 5 evenly spaced points from 0 to 1 (inclusive)
empty = np.empty((2, 2))                       # UNINITIALIZED memory -- fast, but garbage values

print("a:", a)
print("b:\n", b)
print("zeros:\n", zeros)
print("arange:", arange)
print("linspace:", linspace)
print("empty (garbage, don't rely on values):\n", empty)


**`arange` vs. `linspace`:** `arange(start, stop, step)` you specify the *step*; `linspace(start,
stop, num)` you specify the *count* and it's inclusive of `stop`. `linspace` is the safer choice
when floating-point step accumulation could drift (`arange(0, 1, 0.1)` can produce 10 or 11
elements depending on rounding; `linspace(0, 1, 11)` is always exactly 11).

**`np.empty`** skips zero-initialization for speed — only use it when you're about to overwrite
every element anyway (e.g. filling a preallocated buffer in a loop).


## 3. dtype and casting

Every array has exactly one `dtype` for all its elements. This project stores trajectory windows
as `float32` specifically (`scripts/prepare_data.py`, `X_windows.npy`) — half the memory of the
NumPy default `float64`, which matters at 285,926 windows × 20 × 13 floats.


In [ ]:
import numpy as np

default_float = np.array([1.0, 2.0, 3.0])
print("default dtype:", default_float.dtype)          # float64 on most platforms

f32 = np.array([1.0, 2.0, 3.0], dtype=np.float32)
print("explicit dtype:", f32.dtype, "itemsize:", f32.itemsize, "bytes")

# Mixing dtypes upcasts to the "widest" common type automatically.
mixed = np.array([1, 2.5, 3])                          # one float -> whole array becomes float64
print("mixed int+float upcasts to:", mixed.dtype)

int_arr = np.array([1, 2, 3], dtype=np.int32)
as_float = int_arr.astype(np.float32)                  # explicit cast, always makes a copy
print("astype copy:", as_float.dtype, "same memory as int_arr?", np.shares_memory(int_arr, as_float))

# Integer overflow silently wraps -- NumPy does NOT raise, unlike Python's arbitrary-precision ints.
small = np.array([127], dtype=np.int8)
print("int8 127 + 1 wraps to:", (small + 1)[0])         # -128, not 128


## 4. Shape, dimensions, reshaping

`X_windows.npy` has shape `(285926, 20, 13)`: 3 dimensions (`ndim=3`), read outer to inner —
285,926 windows, each 20 timesteps, each timestep 13 features.


In [ ]:
import numpy as np

x = np.arange(24)
print("x.shape:", x.shape, "x.ndim:", x.ndim, "x.size:", x.size)

grid = x.reshape(4, 6)                 # must have the same total size (24 = 4*6)
print("reshaped to (4, 6):\n", grid)

grid3d = x.reshape(2, 3, 4)            # (2, 3, 4): 2*3*4 = 24
print("reshaped to (2, 3, 4), shape:", grid3d.shape)

flat = grid3d.reshape(-1)              # -1 means "infer this dimension"
print("flattened back, shape:", flat.shape)

# reshape(-1, N) is the idiom for "collapse everything except the last axis" --
# e.g. turning (285926, 20, 13) windows into (285926*20, 13) individual steps.
steps_view = grid3d.reshape(-1, 4)
print("collapsed outer dims, shape:", steps_view.shape)

row = np.array([1, 2, 3])
print("row shape:", row.shape)                          # (3,) -- 1D, neither row nor column
col = row[:, np.newaxis]                                  # add an axis -> (3, 1) column vector
print("as column, shape:", col.shape)
print("np.expand_dims does the same:", np.expand_dims(row, axis=0).shape)   # -> (1, 3)
print("squeeze removes size-1 axes:", col.squeeze().shape)  # back to (3,)


**`reshape` vs. `ravel`/`flatten`:** `reshape` returns a *view* when possible (no data copied);
`.flatten()` always copies; `.ravel()` returns a view when possible, otherwise copies (like
`reshape(-1)`). Prefer `reshape`/`ravel` unless you specifically need an independent copy.


## 5. Indexing and slicing

Same syntax as Python lists, extended to multiple dimensions with commas, and — critically —
**basic slices are views**, not copies (see §7).


In [ ]:
import numpy as np

a = np.arange(10)
print("a[3]:", a[3])
print("a[-1]:", a[-1])                 # last element
print("a[2:7]:", a[2:7])               # slice, stop-exclusive like Python
print("a[::2]:", a[::2])               # every other element
print("a[::-1]:", a[::-1])             # reversed

grid = np.arange(20).reshape(4, 5)
print("grid:\n", grid)
print("grid[1, 3]:", grid[1, 3])       # row 1, column 3 -- NOT grid[1][3] (works, but slower/copies)
print("grid[1, :]:", grid[1, :])       # whole row 1
print("grid[:, 2]:", grid[:, 2])       # whole column 2
print("grid[1:3, 2:4]:\n", grid[1:3, 2:4])  # sub-block

# The window-tensor idiom: X[:, 0, :] pulls "the first timestep of every window" -- one line,
# no loop, because slicing with ':' on one axis while indexing another is fully supported.
toy_windows = np.arange(4 * 20 * 13).reshape(4, 20, 13)
first_steps = toy_windows[:, 0, :]
print("first_steps shape (4 windows' step-0):", first_steps.shape)


## 6. Fancy indexing and boolean masking

Two indexing styles that go *beyond* slices: indexing with an array of integers ("fancy"
indexing), and indexing with an array of booleans ("masking"). Both **always copy** — this is the
other half of §7's view-vs-copy story.


In [ ]:
import numpy as np

a = np.arange(10) * 10
idx = [1, 3, 3, 7]                     # indices can repeat and be in any order
print("fancy indexing a[idx]:", a[idx])

mask = a > 50
print("boolean mask:", mask)
print("a[mask]:", a[mask])             # keeps only the True positions

# Assignment works through both, in place:
a[a > 50] = 0
print("after masked assignment:", a)

# 2D fancy indexing selects specific (row, col) pairs when given paired index arrays:
grid = np.arange(20).reshape(4, 5)
rows = [0, 1, 2]
cols = [4, 0, 3]
print("grid[rows, cols] (diagonal-ish picks):", grid[rows, cols])   # grid[0,4], grid[1,0], grid[2,3]

# np.where(condition, if_true, if_false) is the vectorized ternary -- no branching per element.
speed = np.array([1.0, 15.0, 3.0, 60.0, 8.0])
label = np.where(speed > 10, "fast", "slow")
print("np.where labels:", label)


## 7. Views vs. copies — the #1 NumPy gotcha

A **view** shares memory with the original array; mutating one mutates the other. A **copy** is
independent. Getting this wrong is the single most common source of "why did my data change?!"
bugs in NumPy code.

| Operation | View or copy? |
|---|---|
| Basic slicing (`a[1:5]`, `a[:, 2]`) | **View** |
| Fancy indexing (`a[[1,3,5]]`) | **Copy** |
| Boolean masking (`a[a > 0]`) | **Copy** |
| `reshape()` | View *if possible*, else copy |
| `.T` / `np.transpose()` | **View** (just relabels strides) |
| `.copy()` | Always an explicit **copy** |


In [ ]:
import numpy as np

a = np.arange(10)
view = a[2:5]           # a view into a
view[0] = 999            # mutating the view...
print("a after mutating a view of it:", a)   # ...mutates a too! a[2] is now 999

b = np.arange(10)
fancy = b[[2, 3, 4]]     # a copy
fancy[0] = 999
print("b after mutating a fancy-indexed copy:", b)   # b is unchanged

print("view.base is a:", view.base is a)      # views expose their owner via .base
print("fancy.base is b:", fancy.base is b)    # copies have no .base (None)

# The safe habit: if you're not sure, call .copy() explicitly when you need independence.
safe_copy = a[2:5].copy()
safe_copy[0] = -1
print("a unaffected by safe_copy mutation:", a[2])


## 8. Broadcasting, in depth

Broadcasting lets NumPy operate on arrays of *different but compatible* shapes without writing an
explicit loop or copying data to match shapes. This is how `(285926, 20, 13) - (13,)` normalization
works in one line (`scripts/prepare_data.py`'s `Normalizer`).

**The rule**, comparing shapes trailing-dimension-first:
1. If the arrays have different `ndim`, pad the smaller shape's *front* with 1s.
2. Two dimensions are compatible if they're **equal**, or **one of them is 1**.
3. If every dimension pair is compatible, the result shape takes the larger size in each position.
   A size-1 axis is conceptually "stretched" to match — no data is actually duplicated in memory.


In [ ]:
import numpy as np

# (3, 4) + (4,)  ->  the (4,) is treated as (1, 4), stretched across all 3 rows.
grid = np.zeros((3, 4))
row_vec = np.array([1, 2, 3, 4])
print("grid + row_vec:\n", grid + row_vec)

# (3, 1) + (1, 4) -> both get stretched -> (3, 4). This is the classic "outer sum" trick.
col_vec = np.array([[10], [20], [30]])          # shape (3, 1)
print("col_vec + row_vec (outer broadcast):\n", col_vec + row_vec)

# The project's real case: (N, 20, 13) - (13,)
toy_windows = np.random.default_rng(0).normal(size=(4, 20, 13)).astype(np.float32)
feature_mean = toy_windows.mean(axis=(0, 1))     # shape (13,)
print("windows shape:", toy_windows.shape, "mean shape:", feature_mean.shape)
centered = toy_windows - feature_mean            # (13,) broadcasts against the trailing axis
print("centered shape (unchanged):", centered.shape)

# When shapes DON'T broadcast, NumPy raises immediately -- better than a silent wrong answer.
try:
    np.zeros((3, 4)) + np.zeros((3, 5))
except ValueError as e:
    print("broadcast error:", e)


**Using `np.newaxis` to force alignment:** broadcasting aligns from the *trailing* axis, so to
broadcast a `(3,)` array against a `(3, 4)` array's *rows* (not columns), reshape it to `(3, 1)`
explicitly: `arr[:, np.newaxis]`. Getting this backwards is the most common broadcasting bug.


## 9. Vectorized math and ufuncs

A **ufunc** ("universal function") applies elementwise across an array in optimized C, with
broadcasting built in. Nearly every NumPy math function (`np.sin`, `np.exp`, `np.sqrt`, `+`, `*`,
...) is a ufunc.


In [ ]:
import numpy as np

x = np.array([-2.0, -0.5, 0.0, 0.5, 2.0])
print("np.abs:", np.abs(x))
print("np.exp:", np.exp(x).round(3))
print("np.sqrt(abs):", np.sqrt(np.abs(x)).round(3))
print("np.clip to [-1, 1]:", np.clip(x, -1, 1))

# np.hypot(x, y) = sqrt(x**2 + y**2), vectorized -- used for speed magnitude from spd_x/spd_y.
spd_x = np.array([3.0, 0.0, 5.0])
spd_y = np.array([4.0, 2.0, 12.0])
print("speed magnitude:", np.hypot(spd_x, spd_y))

# np.select: vectorized multi-branch "if/elif/elif/else", cleaner than nested np.where.
speed = np.array([1.0, 15.0, 35.0, 8.0])
labels = np.select(
    [speed < 5, speed < 20, speed >= 20],
    ["idle", "cruising", "fast"],
    default="unknown",
)
print("np.select labels:", labels)

# ufuncs support an `out=` parameter to write in place -- avoids allocating a new array.
result = np.empty_like(x)
np.multiply(x, 10, out=result)
print("in-place multiply via out=:", result)


## 10. Aggregations and the `axis` argument

Every reduction (`sum`, `mean`, `std`, `min`, `max`, `argmin`, `argmax`, `any`, `all`, ...) takes an
`axis` argument: which dimension(s) to collapse. No `axis` collapses everything to one scalar.


In [ ]:
import numpy as np

toy_windows = np.random.default_rng(0).normal(size=(4, 20, 13)).astype(np.float32)

print("overall mean (scalar):", toy_windows.mean())
print("mean over axis=0 (average window), shape:", toy_windows.mean(axis=0).shape)      # (20, 13)
print("mean over axis=(0,1) (per-feature), shape:", toy_windows.mean(axis=(0, 1)).shape)  # (13,)
print("mean over axis=-1 (per-step, across features), shape:", toy_windows.mean(axis=-1).shape)  # (4, 20)

# keepdims=True keeps the collapsed axis as size-1 instead of removing it -- makes the result
# broadcast-compatible with the original array (handy for re-centering without a reshape).
mean_kept = toy_windows.mean(axis=(0, 1), keepdims=True)
print("keepdims shape:", mean_kept.shape)   # (1, 1, 13), broadcasts directly against (4, 20, 13)

# argmax/argmin return the INDEX of the extreme value, not the value itself.
per_window_max_step = toy_windows[..., 0].argmax(axis=1)  # index of the highest pos_x per window
print("argmax per window (step index):", per_window_max_step)


## 11. Stacking, concatenating, splitting

Combining arrays: `concatenate` joins along an *existing* axis; `stack` creates a *new* axis;
`vstack`/`hstack` are shortcuts for the common 2D cases.


In [ ]:
import numpy as np

a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

print("concatenate (same ndim, joins existing axis):", np.concatenate([a, b]))
print("stack (adds a NEW axis):\n", np.stack([a, b]))          # shape (2, 3)
print("stack axis=1:\n", np.stack([a, b], axis=1))              # shape (3, 2), interleaved

grid_a = np.zeros((2, 3))
grid_b = np.ones((2, 3))
print("vstack (row-wise):\n", np.vstack([grid_a, grid_b]))       # (4, 3)
print("hstack (column-wise):\n", np.hstack([grid_a, grid_b]))    # (2, 6)

# split is the inverse: divide one array into N equal (or ragged, with array_split) pieces.
whole = np.arange(12)
parts = np.split(whole, 3)
print("split into 3:", [p.tolist() for p in parts])


## 12. Linear algebra

`@` (or `np.matmul`) is matrix multiplication; `*` is elementwise. Confusing the two is a very
common bug once arrays get 2D+.


In [ ]:
import numpy as np

A = np.array([[1.0, 2.0], [3.0, 4.0]])
B = np.array([[5.0, 6.0], [7.0, 8.0]])

print("A * B (elementwise):\n", A * B)
print("A @ B (matrix multiply):\n", A @ B)
print("A.T (transpose, a VIEW):\n", A.T)

v = np.array([1.0, 2.0])
print("A @ v (matrix-vector product):", A @ v)
print("np.linalg.norm(v):", np.linalg.norm(v))          # Euclidean length
print("np.linalg.det(A):", np.linalg.det(A))
print("np.linalg.inv(A):\n", np.linalg.inv(A))

# einsum: one notation for arbitrary contractions -- e.g. dot products per-row without a loop.
X = np.random.default_rng(0).normal(size=(4, 3))
Y = np.random.default_rng(1).normal(size=(4, 3))
row_dots = np.einsum("ij,ij->i", X, Y)     # sum over j, keep i -- one dot product per row
print("einsum row-wise dot products:", row_dots.round(3))
print("same via (X * Y).sum(axis=1):", (X * Y).sum(axis=1).round(3))


## 13. The `random` module

Modern NumPy (≥1.17) prefers the **Generator** API (`np.random.default_rng`) over the legacy
`np.random.seed()` / `np.random.rand()` functions — it's reproducible, thread-safe, and every
demo array in this notebook uses it.


In [ ]:
import numpy as np

rng = np.random.default_rng(seed=42)   # seed once, reuse the generator object

print("uniform [0, 1):", rng.random(3).round(3))
print("normal(0, 1):", rng.normal(size=3).round(3))
print("normal(mean=5, std=2):", rng.normal(loc=5, scale=2, size=3).round(3))
print("integers [0, 10):", rng.integers(0, 10, size=5))
print("choice without replacement:", rng.choice(10, size=3, replace=False))
print("shuffle (in place):", end=" ")
arr = np.arange(5)
rng.shuffle(arr)
print(arr)

# Same seed -> same sequence, every time -- the basis of reproducible experiments (CLAUDE.md rule 4).
rng_a = np.random.default_rng(0)
rng_b = np.random.default_rng(0)
print("same seed reproducible:", np.array_equal(rng_a.normal(size=5), rng_b.normal(size=5)))


## 14. Sorting and searching


In [ ]:
import numpy as np

a = np.array([3, 1, 4, 1, 5, 9, 2, 6])
print("sort (ascending, copy):", np.sort(a))
print("argsort (indices that would sort):", np.argsort(a))
print("original a unchanged:", a)

a.sort()          # in-place variant -- note: mutates a, no return value
print("after a.sort() in place:", a)

sorted_arr = np.array([1, 3, 5, 7, 9])
print("searchsorted(6) insert index:", np.searchsorted(sorted_arr, 6))   # where 6 would insert

dup = np.array([1, 2, 2, 3, 3, 3, 4])
values, counts = np.unique(dup, return_counts=True)
print("unique values:", values, "counts:", counts)


## 15. Performance: vectorization vs. loops

The rule of thumb: **if you're writing a Python `for` loop over array elements, there's almost
certainly a vectorized way to do it that's 10–100x faster.** A few of the most common patterns.


In [ ]:
import numpy as np
import time

n = 500_000
a = np.random.default_rng(0).normal(size=n)

def loop_version(arr):
    out = np.empty_like(arr)
    for i in range(len(arr)):
        out[i] = arr[i] ** 2 if arr[i] > 0 else 0.0
    return out

def vectorized_version(arr):
    return np.where(arr > 0, arr ** 2, 0.0)

t0 = time.perf_counter(); loop_version(a); t_loop = time.perf_counter() - t0
t0 = time.perf_counter(); vectorized_version(a); t_vec = time.perf_counter() - t0
print(f"loop:       {t_loop*1000:.1f} ms")
print(f"vectorized: {t_vec*1000:.1f} ms  ({t_loop/t_vec:.0f}x faster)")
assert np.allclose(loop_version(a), vectorized_version(a))  # allclose, not array_equal -- see §18 on float ==


## 16. Saving and loading

`.npy` stores one array (binary, exact dtype/shape preserved); `.npz` bundles several named arrays
into one file. This is exactly how `data/prepared_data/X_windows.npy`,
`idx_train.npy`/`idx_val.npy`/`idx_test.npy` etc. are stored.


In [ ]:
import numpy as np
import tempfile, pathlib

with tempfile.TemporaryDirectory() as tmp:
    tmp = pathlib.Path(tmp)
    a = np.arange(12).reshape(3, 4)

    np.save(tmp / "a.npy", a)
    loaded = np.load(tmp / "a.npy")
    print("round-trip equal:", np.array_equal(a, loaded))

    np.savez(tmp / "bundle.npz", first=a, second=a.T)
    bundle = np.load(tmp / "bundle.npz")
    print("npz keys:", list(bundle.keys()))
    print("bundle['second'] shape:", bundle["second"].shape)

    # mmap_mode="r" maps the file lazily instead of loading it all into RAM -- this is exactly how
    # scripts/export_simulation_sample.py's WindowStore opens the real 297 MB X_windows.npy.
    mapped = np.load(tmp / "a.npy", mmap_mode="r")
    print("mmap array type:", type(mapped).__name__)


## 17. Worked example: this project's window tensor

Putting §2–§10 together to reproduce, end to end, what `scripts/prepare_data.py`'s `Normalizer`
and the simulator's kinematics computations actually do to a batch of windows.


In [ ]:
import numpy as np

rng = np.random.default_rng(0)
# 5 windows, 20 timesteps, 13 features -- same convention as the real X_windows.npy.
X = rng.normal(loc=0.0, scale=3.0, size=(5, 20, 13)).astype(np.float32)
X[:, :, 8] = np.abs(rng.normal(loc=1.0, scale=0.1, size=(5, 20)))  # column 8 = dt, keep positive

FEATURES = ["pos_x", "pos_y", "spd_x", "spd_y", "acl_x", "acl_y", "hed_x", "hed_y",
            "dt", "dpos_x", "dpos_y", "dspd_x", "dspd_y"]
idx = {name: i for i, name in enumerate(FEATURES)}

# 1. Fit z-score stats on a "train split" (here: all 5 windows, for demo purposes).
mean = X.mean(axis=(0, 1))
std = X.std(axis=(0, 1))
std_safe = np.where(std == 0, 1, std)                 # guard div-by-zero, same as Normalizer
X_norm = (X - mean) / std_safe

print("z-scored: per-feature mean ~0:", X_norm.mean(axis=(0, 1)).round(2))
print("z-scored: per-feature std  ~1:", X_norm.std(axis=(0, 1)).round(2))

# 2. Per-step speed magnitude and implied speed from displacement, vectorized over every
#    window and every step at once -- no Python loop, matching KinematicsSeries.compute() in
#    simulation/src/core/kinematics.ts (just NumPy here instead of TypeScript).
spd = np.hypot(X[:, :, idx["spd_x"]], X[:, :, idx["spd_y"]])
dpos = np.hypot(X[:, :, idx["dpos_x"]], X[:, :, idx["dpos_y"]])
dt = X[:, :, idx["dt"]]
implied_speed = dpos / dt
residual = np.abs(implied_speed - spd)

print("\nclaimed speed shape:", spd.shape, "(5 windows x 20 steps)")
print("mean residual |implied - claimed| speed:", residual.mean().round(3), "m/s")

# 3. Flag physically implausible steps -- the same 60 m/s teleport threshold the simulator uses.
teleport = implied_speed > 60
print("teleport-flagged steps:", teleport.sum(), "of", teleport.size)


## 18. Common pitfalls, summarized

- **Mutating a view you thought was a copy** (§7) — when in doubt, `.copy()` explicitly.
- **Broadcasting against the wrong axis** (§8) — use `np.newaxis` / `keepdims=True` to be explicit
  about which axis stays aligned; don't rely on accidental shape compatibility.
- **Integer overflow silently wrapping** (§3) — NumPy doesn't raise on int8/int16/int32 overflow
  the way Python ints don't overflow at all; pick a wide-enough dtype up front.
- **`==` on floats** — `0.1 + 0.2 == 0.3` is `False` due to floating-point representation; use
  `np.isclose()` / `np.allclose()` for float comparisons, as this project's own tests do
  (`scripts/tests/test_prepare_data.py` uses `pytest.approx`, the same idea).
- **Forgetting `axis=`** — `arr.sum()` with no axis collapses *everything* to one scalar; almost
  always you want `axis=0`, `axis=1`, or a tuple like `axis=(0, 1)`.
- **`np.array_equal` vs. `==`** — `a == b` returns an elementwise boolean array (which is truthy
  only if it has exactly one element); to compare *whole arrays*, use `np.array_equal(a, b)` or
  `np.allclose(a, b)`.
